# IOAI — 2026 Mock Synthetic Speech (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request, gzip, shutil
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-mock-synthetic-speech'
for f in ['X_test.npy','train.csv','test_list.csv']:
    if not os.path.exists(f'data/{f}'): urllib.request.urlretrieve(f'{BASE}/{f}', f'data/{f}')
if not os.path.exists('data/X_train.npy'):
    with open('xt.gz','wb') as out:
        for part in ['Xsp_part_00','Xsp_part_01']:
            urllib.request.urlretrieve(f'{BASE}/{part}', part)
            with open(part,'rb') as p: shutil.copyfileobj(p, out)
    with gzip.open('xt.gz','rb') as g, open('data/X_train.npy','wb') as o: shutil.copyfileobj(g, o)
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Synthetic Speech Detection — 합성음성 탐지 (APOAI 2026 Mock)

**멜 스펙트로그램** `[128,94]` 으로 합성음성(spoof=1)과 실제 사람 음성(bonafide=0)을 이진분류한다.
`data/X_train.npy`+`data/train.csv`(filename,label)로 학습, `data/X_test.npy`(+`test_list.csv`)를 예측해
`submission.csv`(filename,label). 채점: binary **F1**(pos=spoof). (사이트는 균형 서브샘플: train 3000 / val 800.)

In [ ]:
import numpy as np, pandas as pd
Xtr = np.load("data/X_train.npy").astype("float32")   # [N,128,94]
train = pd.read_csv("data/train.csv")                 # filename, label
Xte = np.load("data/X_test.npy").astype("float32")
test_files = pd.read_csv("data/test_list.csv")["filename"].tolist()
print(Xtr.shape, "| label dist", train.label.value_counts().to_dict())

In [ ]:
# 베이스라인: 전부 bonafide(0) — F1(pos=spoof)=0
pd.DataFrame({"filename": test_files, "label": [0]*len(test_files)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(test_files))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)